# Job Matching Engine — Data Exploration & TF-IDF Baseline

Pipeline: Kaggle dataset + manually collected German job postings -> cleaning -> bilingual TF-IDF matching against my CV.

In [1]:
import os

# Ensure working directory is the project root, not notebooks/
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')

print("Working directory:", os.getcwd())

Working directory: C:\Users\Administrator\IdeaProjects\job-matching-engine


## 1. Load Kaggle dataset

In [2]:
import os
import pandas as pd

if not os.path.exists('data/raw/job_postings.csv'):
    import kaggle
    kaggle.api.dataset_download_files(
        'asaniczka/data-science-job-postings-and-skills',
        path='data/raw', unzip=True
    )

postings = pd.read_csv('data/raw/job_postings.csv')
skills = pd.read_csv('data/raw/job_skills.csv')
summary = pd.read_csv('data/raw/job_summary.csv')

print(postings.shape, skills.shape, summary.shape)

(12217, 15) (12217, 2) (12217, 2)


In [3]:
df = postings.merge(skills, on='job_link', how='left') \
    .merge(summary, on='job_link', how='left')

print(df.shape)
print(df.columns.tolist())
df.head(2)

(12217, 17)
['job_link', 'last_processed_time', 'last_status', 'got_summary', 'got_ner', 'is_being_worked', 'job_title', 'company', 'job_location', 'first_seen', 'search_city', 'search_country', 'search_position', 'job_level', 'job_type', 'job_skills', 'job_summary']


,job_link,last_processed_time,last_status,got_summary,got_ner,is_being_worked,job_title,company,job_location,first_seen,search_city,search_country,search_position,job_level,job_type,job_skills,job_summary
0,https://www.linkedin.com/jobs/view/senior-mach...,2024-01-21 08:08:48.031964+00,Finished NER,t,t,f,Senior Machine Learning Engineer,Jobs for Humanity,"New Haven, CT",2024-01-14,East Haven,United States,Agricultural-Research Engineer,Mid senior,Onsite,"Machine Learning, Programming, Python, Scala, ...",Company Description\nJobs for Humanity is part...
1,https://www.linkedin.com/jobs/view/principal-s...,2024-01-20 04:02:12.331406+00,Finished NER,t,t,f,"Principal Software Engineer, ML Accelerators",Aurora,"San Francisco, CA",2024-01-14,El Cerrito,United States,Set-Key Driver,Mid senior,Onsite,"C++, Python, PyTorch, TensorFlow, MXNet, CUDA,...",Who We Are\nAurora (Nasdaq: AUR) is delivering...


## 2. Explore the data

In [4]:
# Check missing values per column
print(df.isnull().sum())

# Check distribution of job titles
print(df['job_title'].value_counts().head(20))

# Preview raw job summary text
print(df.loc[0, 'job_summary'])

job_link               0
last_processed_time    0
last_status            0
got_summary            0
got_ner                0
is_being_worked        0
job_title              0
company                0
job_location           1
first_seen             0
search_city            0
search_country         0
search_position        0
job_level              0
job_type               0
job_skills             5
job_summary            0
dtype: int64
job_title
Senior Data Engineer                                        285
Senior Data Analyst                                         163
Data Engineer                                               149
Senior MLOps Engineer                                       138
Data Analyst                                                137
Data Scientist                                              128
Lead Data Engineer                                          123
Senior Data Scientist                                       119
Data Architect                          

## 3. Clean the data

Drop missing critical fields and exact-duplicate postings.

In [5]:
# Drop rows with missing critical fields
df = df.dropna(subset=['job_location', 'job_skills']).reset_index(drop=True)

# Check for duplicate postings
print(f"Duplicate job_links: {df['job_link'].duplicated().sum()}")
df = df.drop_duplicates(subset=['job_link']).reset_index(drop=True)

print(df.shape)

Duplicate job_links: 0
(12211, 17)


### Text cleaning function

Strips boilerplate (legal/diversity notices, application instructions) from the tail of each posting.
Handles **both English and German** markers, since the final dataset combines Kaggle (EN) postings with manually collected German postings.

Only strips boilerplate found in the last 15% of the text.  An earlier version stripped from the *first* occurrence anywhere in the text, which accidentally wiped out entire postings when a marker like "reasonable accommodations" appeared early in a legitimate description (found via a validation check, see below).

In [6]:
import re

def clean_job_summary(text):
    if not isinstance(text, str):
        return ""

    # Only strip boilerplate if it appears in the last 15% of the text
    # Bilingual boilerplate markers (English + German)
    boilerplate_markers = [
        "show more", "equal opportunity employer",
        "protected veteran status", "reasonable accommodations",
        "chancengleichheit", "wir freuen uns auf deine bewerbung",
        "bewerbungsunterlagen", "vielfalt und inklusion",
    ]

    cutoff_point = int(len(text) * 0.85)
    tail = text[cutoff_point:].lower()

    for marker in boilerplate_markers:
        idx = text.lower().find(marker, cutoff_point)
        if idx != -1:
            text = text[:idx]
            break

    text = re.sub(r"\s+", " ", text).strip()
    return text

df['job_summary_clean'] = df['job_summary'].apply(clean_job_summary)

print(df['job_summary_clean'].str.len().describe())

count    12211.000000
mean      4256.025469
std       2293.421298
min         21.000000
25%       2541.500000
50%       3972.000000
75%       5708.000000
max      19177.000000
Name: job_summary_clean, dtype: float64


In [7]:
# Validation: check no posting got wiped to empty by cleaning
empty_after_clean = df[df['job_summary_clean'].str.len() == 0]
print(f"Number of rows now empty: {len(empty_after_clean)}")

Number of rows now empty: 0


Remove non-relevant administrative/procedural postings and reposted duplicates (same title + company, different link — common with recruiting agencies reposting the same listing).

In [8]:
# Flag postings that are mostly procedural/administrative
admin_keywords = ['CalCareer', 'Examination/Employment Application', 'Statement of Qualifications']
df['is_admin_posting'] = df['job_summary'].str.contains('|'.join(admin_keywords), case=False, na=False)

print(f"Admin/procedural postings detected: {df['is_admin_posting'].sum()}")

df_filtered = df[~df['is_admin_posting']].reset_index(drop=True)

# Remove reposted duplicates (same title + company, different job_link)
before = df_filtered.shape[0]
df_filtered = df_filtered.drop_duplicates(subset=['job_title', 'company']).reset_index(drop=True)
after = df_filtered.shape[0]
print(f"Removed {before - after} duplicate postings (same title + company)")

print(df_filtered.shape)

Admin/procedural postings detected: 24
Removed 3363 duplicate postings (same title + company)
(8824, 19)


## 4. Load my CV (reference text for matching)

In [9]:
with open('data/my_cv.txt', 'r', encoding='utf-8') as f:
    my_cv = f.read()

print(f"CV length: {len(my_cv)} characters")
print(my_cv[:300])

CV length: 4692 characters
﻿Pharel Harold Nanseu Kombou

Data Science, Machine Learning & MLOps – Werkstudent / Praktikant

Gießen, Deutschland | 0162571365 | haroldpharel@gmail.com | linkedin.com/in/pharel-nanseu-042281356 | github.com/Pharel8

PROFIL

Informatik-Student (B.Sc., THM Gießen, 5. Semester) mit Schwerpunkt Data 


## 5. Load manually collected German job postings

10 real Werkstudent/Praktikum postings (LinkedIn/StepStone/Xing) collected to complement the Kaggle dataset, which is US/English-only.

In [10]:
manual_offers = pd.read_csv('data/manual_offers.csv')
manual_offers.columns = manual_offers.columns.str.strip()
manual_offers = manual_offers.dropna(subset=['job_title', 'company']).reset_index(drop=True)
manual_offers['company'] = manual_offers['company'].str.strip()
manual_offers['job_summary_clean'] = manual_offers['job_summary'].apply(clean_job_summary)

print(manual_offers.shape)
print(manual_offers['job_summary_clean'].str.len().describe())

(10, 7)
count      10.000000
mean     1137.300000
std       347.241847
min       804.000000
25%       964.750000
50%      1083.000000
75%      1149.250000
max      2027.000000
Name: job_summary_clean, dtype: float64


## 6. Combine Kaggle + manual offers into one dataset

In [11]:
combined = pd.concat([
    df_filtered[['job_title', 'company', 'job_location', 'job_summary_clean', 'job_skills']],
    manual_offers[['job_title', 'company', 'job_location', 'job_summary_clean', 'job_skills']]
], ignore_index=True)

print(f"Kaggle offers: {len(df_filtered)}")
print(f"Manual offers: {len(manual_offers)}")
print(f"Combined total: {len(combined)}")

Kaggle offers: 8824
Manual offers: 10
Combined total: 8834


## 7. TF-IDF matching — bilingual stopwords

**Finding during development:** using only English stopwords (`stop_words='english'`) with a German CV against this bilingual corpus caused generic German HR vocabulary ("und", "für", "mit"...) to dominate the similarity score. This produced a very high top score (~0.82) but pulled in false positives — e.g. "Teamleiter Produktion" and "Projektleiter Großschaden" ranked highly purely because they were in German, not because they were relevant to a Data Science profile.

Adding German stopwords alongside English ones fixed this: scores dropped to a more realistic ~0.44, and the false positives disappeared — the top results became consistently ML/Data Science roles.

In [12]:
import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords

english_stopwords = set(stopwords.words('english'))
german_stopwords = set(stopwords.words('german'))
combined_stopwords = list(english_stopwords | german_stopwords)

print(f"Total combined stopwords: {len(combined_stopwords)}")

Total combined stopwords: 424


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Administrator\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [13]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Combine CV with all job summaries into one corpus
corpus = [my_cv] + combined['job_summary_clean'].tolist()

vectorizer = TfidfVectorizer(stop_words=combined_stopwords, max_features=5000)
tfidf_matrix = vectorizer.fit_transform(corpus)

# CV is the first row (index 0), compare it against all job postings (index 1 onward)
cv_vector = tfidf_matrix[0:1]
job_vectors = tfidf_matrix[1:]

similarity_scores = cosine_similarity(cv_vector, job_vectors).flatten()

combined['tfidf_score'] = similarity_scores

# Show top 15 matches
top_matches = combined.sort_values('tfidf_score', ascending=False).head(15)
print(top_matches[['job_title', 'company', 'tfidf_score']])

                                              job_title  \
8825           Werkstudent (m/w/d) Data Science - Hotel   
8833       Werkstudent Data Analytics / Computer Vision   
8832        Werkstudent Data Engineering & Data Science   
8831  WerkstudentIn Data Science, Machine Learning & AI   
5811                          Machine Learning Engineer   
6037                 Machine Learning Software Engineer   
8569                              Senior Data Scientist   
1105                              Senior MLOps Engineer   
3873             Machine Learning Platform Engineer x 2   
421               Senior Machine Learning Engineer - AI   
3725                          Machine Learning Engineer   
3906                 Machine Learning Platform Engineer   
5513                   Senior Machine Learning Engineer   
8826  Werkstudent Data Science & Machine Learning (m...   
6180            ML Engineer - Data & Advanced Analytics   

                                                company

## 8. Multilingual Sentence-Transformer Embeddings

TF-IDF represents text as word-frequency vectors — it has no notion of meaning, only exact word overlap. This is why a German CV against English postings (or vice versa) scored poorly unless language-specific stopwords were added (see Section 7).

**Sentence-Transformers** solve this differently: instead of counting words, a fine-tuned Transformer model encodes each text into a single dense vector that captures its *meaning*. With a multilingual model, semantically equivalent sentences in German and English end up close together in the same vector space — no stopword patching needed.

Model used: `paraphrase-multilingual-mpnet-base-v2` (supports 50+ languages including German and English, strong balance of quality/speed per MTEB benchmarks).



In [14]:
from sentence_transformers import SentenceTransformer

# Multilingual model: handles German + English in the same embedding space
model = SentenceTransformer('paraphrase-multilingual-mpnet-base-v2')

# Encode CV and all job postings
cv_embedding = model.encode([my_cv])
job_embeddings = model.encode(combined['job_summary_clean'].tolist(), show_progress_bar=True)

print(f"CV embedding shape: {cv_embedding.shape}")
print(f"Job embeddings shape: {job_embeddings.shape}")

C:\Users\Administrator\IdeaProjects\job-matching-engine\.venv\Lib\site-packages\sentence_transformers\cross_encoder\CrossEncoder.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

C:\Users\Administrator\IdeaProjects\job-matching-engine\.venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Administrator\.cache\huggingface\hub\models--sentence-transformers--paraphrase-multilingual-mpnet-base-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/277 [00:00<?, ?it/s]

CV embedding shape: (1, 768)
Job embeddings shape: (8834, 768)


### Compare embedding-based matching to the TF-IDF baseline

Both scores are stored side by side in `combined` (`tfidf_score` vs. `embedding_score`), enabling a direct comparison

In [15]:
from sklearn.metrics.pairwise import cosine_similarity

similarity_scores_embeddings = cosine_similarity(cv_embedding, job_embeddings).flatten()
combined['embedding_score'] = similarity_scores_embeddings

top_matches_embeddings = combined.sort_values('embedding_score', ascending=False).head(15)
print(top_matches_embeddings[['job_title', 'company', 'embedding_score']])

                                              job_title  \
2042                      AI/ML Engineer-Data Scientist   
1435                          Machine Learning Engineer   
8604  AI and ML Research and Development Technologis...   
4443                                  Data Engineer III   
6134                              Senior Data Scientist   
5868                          Machine Learning Engineer   
8829  Praktikant*in im Bereich Big Data & Advanced A...   
6337                                Lead Data Scientist   
8825           Werkstudent (m/w/d) Data Science - Hotel   
3659                                     Data Scientist   
1173                Lecturer/Reader in Machine Learning   
8279                          Machine Learning Engineer   
8831  WerkstudentIn Data Science, Machine Learning & AI   
7688  Senior Data Scientist - Large Language Models ...   
329                                Data Scientist (MMM)   

                          company  embedding_score  
20

## 9. Save intermediate results

Embeddings are expensive to recompute (~several minutes for 8834 postings). Save the dataframe with both TF-IDF and embedding scores to disk, so future kernel restarts don't require recomputing them.

In [17]:
combined.to_pickle('data/combined_with_scores.pkl')
print("Saved to data/combined_with_scores.pkl")

Saved to data/combined_with_scores.pkl


## 10. Build a manual annotation set for evaluation

To objectively compare TF-IDF vs. embedding-based matching (Precision@K), a small ground-truth set is needed: a sample of postings manually labeled as relevant (1) or not relevant (0) for my profile.

Sample composition: top 20 by TF-IDF score, top 20 by embedding score (union, deduplicated), plus 10 low-scoring postings as clear negative examples — ensuring the annotation set covers both methods' top picks and some easy negatives.

In [18]:
top_tfidf_ids = combined.sort_values('tfidf_score', ascending=False).head(20).index
top_embedding_ids = combined.sort_values('embedding_score', ascending=False).head(20).index
random_low_ids = combined[combined['embedding_score'] < 0.3].sample(10, random_state=42).index

annotation_ids = list(set(top_tfidf_ids) | set(top_embedding_ids) | set(random_low_ids))
annotation_set = combined.loc[annotation_ids].reset_index(drop=True)

print(f"Total offers to annotate: {len(annotation_set)}")
annotation_set[['job_title', 'company', 'tfidf_score', 'embedding_score']]

Total offers to annotate: 48


,job_title,company,tfidf_score,embedding_score
0,Werkstudent Data Engineering & Data Science,Mercedes-Benz Tech Innovation,0.388076,0.433312
1,Werkstudent Data Analytics / Computer Vision,CemeCon AG,0.394263,0.478142
2,Werkstudent Data Science - Jobsuche & Job-Empf...,XING,0.247093,0.427339
3,Data Scientist with AL & ML,Saransh Inc,0.053670,0.528240
4,Senior Data Scientist - Large Language Models ...,Datadog,0.055635,0.532660
5,Senior Machine Learning Engineer,Lirio,0.280516,0.465927
6,Machine Learning Engineer,Tiger Analytics,0.282055,0.419850
7,Machine Learning Software Engineer,The AI Institute,0.296839,0.343719
8,Lecturer/Reader in Machine Learning,The University of Edinburgh,0.105863,0.541431
9,Machine Learning Engineer,Optimal Staffing,0.106581,0.580845


### Export for manual annotation

Exported as CSV for manual review — the `relevant` column is filled in by hand (1 = relevant to my profile, 0 = not relevant) before computing Precision@K.

In [19]:
annotation_set['relevant'] = None

annotation_set[['job_title', 'company', 'job_summary_clean', 'tfidf_score', 'embedding_score', 'relevant']].to_csv(
    'data/annotation_set.csv', index=False, encoding='utf-8'
)
print("Exported to data/annotation_set.csv — fill in the 'relevant' column (1 or 0)")

Exported to data/annotation_set.csv — fill in the 'relevant' column (1 or 0)


In [20]:
annotation_set = pd.read_csv('data/annotation_set.csv')
print(annotation_set.shape)
print(annotation_set['relevant'].value_counts())

(48, 6)
relevant
0    25
1    23
Name: count, dtype: int64


## 11. Compute Precision@K — TF-IDF vs. Embeddings

Load the manually annotated set and compare both matching methods objectively: for each K, what fraction of the top-K results (by score) are actually relevant, according to manual annotation?

**Note on methodology:** Precision@K here is computed only over the 48 manually annotated postings (not the full ~8834), since ground-truth labels only exist for this subset. The relative ranking within this subset is a reasonable proxy, but a fully rigorous evaluation would annotate an independent random sample rather than a set built from each method's own top results.

In [21]:
def precision_at_k(df, score_col, k):
    top_k = df.sort_values(score_col, ascending=False).head(k)
    relevant_in_top_k = top_k['relevant'].sum()
    return relevant_in_top_k / k

# Compare both methods at different K values
for k in [5, 10, 15, 20]:
    p_tfidf = precision_at_k(annotation_set, 'tfidf_score', k)
    p_embedding = precision_at_k(annotation_set, 'embedding_score', k)
    print(f"K={k}: TF-IDF Precision@{k} = {p_tfidf:.2f} | Embeddings Precision@{k} = {p_embedding:.2f}")

K=5: TF-IDF Precision@5 = 1.00 | Embeddings Precision@5 = 0.40
K=10: TF-IDF Precision@10 = 0.50 | Embeddings Precision@10 = 0.70
K=15: TF-IDF Precision@15 = 0.53 | Embeddings Precision@15 = 0.67
K=20: TF-IDF Precision@20 = 0.55 | Embeddings Precision@20 = 0.60


In [22]:
top5_tfidf = annotation_set.sort_values('tfidf_score', ascending=False).head(5)
print(top5_tfidf[['job_title', 'company', 'tfidf_score', 'relevant']])

                                            job_title  \
38           Werkstudent (m/w/d) Data Science - Hotel   
1        Werkstudent Data Analytics / Computer Vision   
0         Werkstudent Data Engineering & Data Science   
47  WerkstudentIn Data Science, Machine Learning & AI   
20                          Machine Learning Engineer   

                                              company  tfidf_score  relevant  
38                           CHECK24 Vergleichsportal     0.441015         1  
1                                          CemeCon AG     0.394263         1  
0                       Mercedes-Benz Tech Innovation     0.388076         1  
47                                          SCHOTT AG     0.387450         1  
20  HummingBirds Consulting  LLC - now doing Busin...     0.311738         1  


## 12. Compare embedding models

Test a second, smaller multilingual model (`paraphrase-multilingual-MiniLM-L12-v2`, 384 dimensions) against the one used so far (`paraphrase-multilingual-mpnet-base-v2`, 768 dimensions), evaluated on the same annotated test set for a direct, apples-to-apples comparison.

In [23]:
model_v2 = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

# Encode only the annotated subset (fast enough, no need to re-encode all 8834 postings)
cv_embedding_v2 = model_v2.encode([my_cv])
annotation_embeddings_v2 = model_v2.encode(annotation_set['job_summary_clean'].tolist())

similarity_v2 = cosine_similarity(cv_embedding_v2, annotation_embeddings_v2).flatten()
annotation_set['embedding_score_v2'] = similarity_v2

print(annotation_set[['job_title', 'company', 'embedding_score', 'embedding_score_v2', 'relevant']].head(10))

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

C:\Users\Administrator\IdeaProjects\job-matching-engine\.venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Administrator\.cache\huggingface\hub\models--sentence-transformers--paraphrase-multilingual-MiniLM-L12-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

                                           job_title  \
0        Werkstudent Data Engineering & Data Science   
1       Werkstudent Data Analytics / Computer Vision   
2  Werkstudent Data Science - Jobsuche & Job-Empf...   
3                        Data Scientist with AL & ML   
4  Senior Data Scientist - Large Language Models ...   
5                   Senior Machine Learning Engineer   
6                          Machine Learning Engineer   
7                 Machine Learning Software Engineer   
8                Lecturer/Reader in Machine Learning   
9                          Machine Learning Engineer   

                         company  embedding_score  embedding_score_v2  \
0  Mercedes-Benz Tech Innovation         0.433312            0.522526   
1                     CemeCon AG         0.478142            0.509772   
2                           XING         0.427339            0.552408   
3                    Saransh Inc         0.528240            0.490117   
4                 

In [24]:
for k in [5, 10, 15, 20]:
    p_tfidf = precision_at_k(annotation_set, 'tfidf_score', k)
    p_mpnet = precision_at_k(annotation_set, 'embedding_score', k)
    p_minilm = precision_at_k(annotation_set, 'embedding_score_v2', k)
    print(f"K={k}: TF-IDF={p_tfidf:.2f} | mpnet-base-v2={p_mpnet:.2f} | MiniLM-L12-v2={p_minilm:.2f}")

K=5: TF-IDF=1.00 | mpnet-base-v2=0.40 | MiniLM-L12-v2=0.60
K=10: TF-IDF=0.50 | mpnet-base-v2=0.70 | MiniLM-L12-v2=0.50
K=15: TF-IDF=0.53 | mpnet-base-v2=0.67 | MiniLM-L12-v2=0.60
K=20: TF-IDF=0.55 | mpnet-base-v2=0.60 | MiniLM-L12-v2=0.55


## 13. RAG/LLM Layer — Match Explanations

Generate a natural-language explanation for why a CV matches a given posting, using an LLM (GPT-4o-mini). The prompt is constrained to only reference facts actually present in the CV and job posting text, to avoid hallucinated claims.

In [25]:
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

print("API key loaded:", bool(os.getenv("OPENAI_API_KEY")))

API key loaded: True


In [27]:
def generate_match_explanation(cv_text, job_title, company, job_summary, tfidf_score, embedding_score):
    prompt = f"""You are an assistant that explains why a CV matches a job posting.

IMPORTANT:
- Only mention skills and experience that are LITERALLY present in the CV text below. Do not invent anything.
- Always respond in the SAME LANGUAGE as the CV text below, regardless of the language of the job posting.

CV:
{cv_text}

JOB POSTING (may be in a different language than the CV):
Title: {job_title}
Company: {company}
Description: {job_summary}

Respond in the CV's language, in exactly this format:
MATCHING SKILLS: [list 2-4 concrete overlaps between CV and posting]
MISSING SKILLS: [list 1-3 requirements from the posting NOT found in the CV]
BRIEF ASSESSMENT: [1-2 sentences on whether this match makes sense]
"""

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.3,
        max_tokens=300
    )
    return response.choices[0].message.content

# Test on the #1 embedding match
test_offer = combined.sort_values('embedding_score', ascending=False).iloc[0]

explanation = generate_match_explanation(
    my_cv,
    test_offer['job_title'],
    test_offer['company'],
    test_offer['job_summary_clean'],
    test_offer['tfidf_score'],
    test_offer['embedding_score']
)

print(f"Offer: {test_offer['job_title']} @ {test_offer['company']}")
print(f"Score: {test_offer['embedding_score']:.2f}")
print()
print(explanation)

# Test on a strong German match
schott_offer = combined[combined['company'] == 'SCHOTT AG'].iloc[0]

explanation_schott = generate_match_explanation(
    my_cv,
    schott_offer['job_title'],
    schott_offer['company'],
    schott_offer['job_summary_clean'],
    schott_offer['tfidf_score'],
    schott_offer['embedding_score']
)

print(f"Offer: {schott_offer['job_title']} @ {schott_offer['company']}")
print(f"Score: {schott_offer['embedding_score']:.2f}")
print()
print(explanation_schott)

Offer: AI/ML Engineer-Data Scientist @ Saransh Inc
Score: 0.59

MATCHING SKILLS: Erfahrung in Python, Kenntnisse in Machine Learning und MLOps, praktische Erfahrung im Aufbau vollständiger ML-Workflows, Kenntnisse in Cloud-Grundlagen (aktuell AWS/Azure im Selbststudium).  
MISSING SKILLS: Erfahrung mit LLM und Generative AI, Kenntnisse in Dokumentenextraktion/NLP, Erfahrung mit AWS Neptune oder Neo4J.  
BRIEF ASSESSMENT: Der CV zeigt starke Fähigkeiten in Machine Learning und Python, jedoch fehlen spezifische Erfahrungen mit LLM, Generative AI und den geforderten Cloud-Datenbanken, was die Eignung für die Stelle einschränkt.
Offer: WerkstudentIn Data Science, Machine Learning & AI @ SCHOTT AG
Score: 0.53

MATCHING SKILLS: Informatik-Student mit Schwerpunkt Data Science und Machine Learning; praktische Erfahrung im Aufbau vollständiger ML-Workflows, einschließlich Experiment-Tracking und Monitoring mit Weights & Biases; gute Kenntnisse in Python; Interesse an MLOps und Cloud-Plattformen

### Apply RAG explanations to top N matches automatically

Instead of calling the function manually per offer, generate explanations for the top N postings by embedding score in one batch.

In [28]:
import time

def generate_top_n_explanations(df, n=5, score_col='embedding_score'):
    top_n = df.sort_values(score_col, ascending=False).head(n).copy()
    explanations = []

    for _, row in top_n.iterrows():
        explanation = generate_match_explanation(
            my_cv,
            row['job_title'],
            row['company'],
            row['job_summary_clean'],
            row.get('tfidf_score', None),
            row[score_col]
        )
        explanations.append(explanation)
        time.sleep(1)  # avoid hitting rate limits

    top_n['rag_explanation'] = explanations
    return top_n

top_5_with_explanations = generate_top_n_explanations(combined, n=5)

for _, row in top_5_with_explanations.iterrows():
    print(f"\n{'='*80}")
    print(f"{row['job_title']} @ {row['company']} (score: {row['embedding_score']:.2f})")
    print(f"{'-'*80}")
    print(row['rag_explanation'])


AI/ML Engineer-Data Scientist @ Saransh Inc (score: 0.59)
--------------------------------------------------------------------------------
MATCHING SKILLS: Erfahrung in Python, Kenntnisse in Machine Learning, praktische Erfahrung im Aufbau vollständiger ML-Workflows, Interesse an Cloud-Plattformen (aktuell Vertiefung von Cloud-Grundlagen im Selbststudium).  
MISSING SKILLS: Erfahrung mit LLM und Generative AI, Erfahrung in einem cloud-nativen Umfeld wie AWS, Kenntnisse in der Arbeit mit graphbasierten Ontologien.  
BRIEF ASSESSMENT: Der CV zeigt relevante Fähigkeiten in Machine Learning und Python, jedoch fehlen spezifische Erfahrungen im Bereich LLM, Generative AI und cloud-nativen Umgebungen, was die Passung für die Stelle einschränkt.

Machine Learning Engineer @ Optimal Staffing (score: 0.58)
--------------------------------------------------------------------------------
MATCHING SKILLS: Erfahrung mit Machine Learning-Algorithmen (z.B. CNNs, RNNs) und deren Implementierung, Progr